# BRAWLPIT Packet-Level RL Training (S419/S420)

Runs the real packet-level RL training pipeline (`scripts/rl_env_packet.py` / `scripts/rl_train_packet.py` / `scripts/rl_league.py`) in a normal Python environment with pip access -- this repo's own dev sandbox is externally-managed with no sudo/venv, so `gymnasium`/`stable_baselines3` can't be installed there. Colab (or any real machine) is where this pipeline is actually meant to run.

See `docs/RL_TRAINING_NORTHSTAR.md` for the full design writeup: real UDP wire-protocol observation/action, `--fast-forward`, the PARENA-compiled single-agent fractal commander, the AlphaStar-style three-role league with real Elo, the reward design, and (S420) the real, remote, IDUNA-hosted checkpoint registry this notebook pushes to -- so training from Colab and training from any other machine can contribute to the SAME shared league, not N separate local ones.

**Before running**: this clones a private GitHub repo, so you need a GitHub token with `repo` read access (https://github.com/settings/tokens), and, if you want this run's checkpoints to join the shared registry, the real `BRAWLPIT-RL` agent secret from `IDUNA/var/agent-secrets.env` (`IDUNA_SECRET_BRAWLPIT_RL`). Both are pasted once via `getpass` below -- held only in this notebook's own runtime memory, never written to a cell's saved output or committed anywhere.

In [ ]:
import getpass
github_token = getpass.getpass("GitHub personal access token (repo read scope): ")
iduna_agent_secret = getpass.getpass("BRAWLPIT-RL agent secret (blank to skip the remote registry): ")

In [ ]:
# Clone BRAWLPIT. commander_mod.c (PARENA's own compiled output) is already checked in --
# no need to clone or build PARENA itself for training.
!git clone https://{github_token}@github.com/emilyspringerton/BRAWLPIT.git
%cd BRAWLPIT
del github_token  # real, deliberate -- don't keep the token in memory longer than the clone needs it

In [ ]:
# Colab's own base image already ships gcc/build-essential -- this is a real, harmless no-op
# safety net for a from-scratch machine, not assumed-necessary busywork.
!apt-get -qq update && apt-get -qq install -y build-essential

In [ ]:
# Builds bin/brawlpit_server (with --fast-forward) and build/libbrawlpit_commander.so.
!chmod +x scripts/build_training.sh
!./scripts/build_training.sh

In [ ]:
!pip install -q gymnasium stable-baselines3

## Sanity check: real UDP wire protocol round trip

Starts a real `bin/brawlpit_server` and runs `rl_env_packet.py --smoke-test` against it -- no `gymnasium`/`stable_baselines3` needed for this part, just confirms the packet plumbing (handshake, snapshot decode, commander posture, reward) actually works in THIS environment before spending any real training compute.

In [ ]:
import subprocess, time
server = subprocess.Popen(["./bin/brawlpit_server", "--fast-forward"])
time.sleep(1)
!python3 scripts/rl_env_packet.py --smoke-test --steps 10
server.terminate()
server.wait(timeout=5)

## Check the shared remote registry (S420)

Real, live standings from IDUNA's own checkpoint registry (`IDUNA/internal/brawlpit/checkpoint_store.go`) before this run adds anything -- shows checkpoints pushed from THIS box, prior Colab runs, or anywhere else that's authenticated as `BRAWLPIT-RL`. List/download need no auth (same trust level `GET /api/v1/brawlpit-levels` already established).

In [ ]:
IDUNA_BASE_URL = "https://okemily.com"  # change if training against a different IDUNA instance
!python3 scripts/rl_registry.py list --base-url {IDUNA_BASE_URL}

## Real training run

Runs all three league archetypes (Main / Main Exploiter / League Exploiter) together via `rl_train_packet.py` -- each generation's checkpoint save registers all three into the local league (`scripts/rl_league.py`'s own `register_generation_snapshot`) AND, when `--registry-agent-secret` is set, pushes all three to IDUNA's real shared registry, tagged with `--registry-source-location colab` so it's visible where each one came from.

Start small (`--total-timesteps` in the low thousands) to confirm a full save/register/push cycle completes end to end before committing real GPU/CPU time to a long run.

In [ ]:
import os
env = os.environ.copy()
if iduna_agent_secret:
    env["IDUNA_AGENT_SECRET"] = iduna_agent_secret

cmd = [
    "python3", "scripts/rl_train_packet.py",
    "--total-timesteps", "5000",
    "--save-freq", "2500",
    "--league-dir", "league_data",
    "--output-dir", "rl_packet_checkpoints",
]
if iduna_agent_secret:
    cmd += ["--registry-url", IDUNA_BASE_URL, "--registry-source-location", "colab"]

import subprocess
subprocess.run(cmd, env=env, check=True)

## Inspect the league

Local Elo standings from this run's own `league_data/` (fast, no network), then the real shared registry again (if you pushed) -- confirming this run's own checkpoints actually joined it.

In [ ]:
import sys
sys.path.insert(0, "scripts")
from rl_league import LeagueManager

league = LeagueManager("league_data")
print("-- local league_data --")
for m in sorted(league.all_members(), key=lambda m: (m.role, m.generation)):
    print(f"{m.role:18s} gen={m.generation:3d}  elo={league.get_elo(m.id):7.1f}  {m.path}")

In [ ]:
if iduna_agent_secret:
    print("-- shared remote registry --")
    !python3 scripts/rl_registry.py list --base-url {IDUNA_BASE_URL}
else:
    print("no agent secret was provided -- this run's checkpoints stayed local only.")

## Download results

Zips the local league registry + checkpoints so they survive past this Colab runtime (which is ephemeral -- nothing here persists once the runtime recycles). If you pushed to the shared remote registry above, those checkpoints already survive independently of this runtime too -- pull them back later with `python3 scripts/rl_registry.py pull --base-url <url> <id> <dest>`.

In [ ]:
!zip -qr brawlpit_rl_results.zip league_data rl_packet_checkpoints
from google.colab import files
files.download("brawlpit_rl_results.zip")